# Telco Customer Churn — Analysis Notebook

This notebook is generated by `scripts/build_notebook.py` and executed end to end, so
the outputs below are reproducible artefacts of the pipeline rather than saved manual runs.

1. Load and validate the raw extract
2. Clean and document data-quality issues
3. Explore the target, numeric and categorical structure
4. Review the trained model's hold-out performance


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from churn import config, data, eda
from churn.clean import clean

plt.rcParams["figure.figsize"] = (10, 4.5)
pd.set_option("display.max_columns", 50)


## 1. Load and validate

In [2]:
raw = data.load_raw()
validation = data.validate(raw)
print(f"rows={validation['rows']:,}  columns={validation['columns']}")
print(raw[["tenure", "MonthlyCharges", "TotalCharges", "Churn"]].head())


rows=7,043  columns=21
   tenure  MonthlyCharges TotalCharges Churn
0       1           29.85        29.85    No
1      34           56.95       1889.5    No
2       2           53.85       108.15   Yes
3      45           42.30      1840.75    No
4       2           70.70       151.65   Yes


## 2. Cleaning log

In [3]:
cleaned, log = clean(raw)
pd.Series(log, name="value").to_frame()


,value
rows_in,7043
duplicate_customer_ids_removed,0
rows_missing_target_dropped,0
total_charges_zero_for_new_customers,11
numeric_median_imputed,{}
categorical_unknown_imputed,{}
remaining_nulls,0
high_spend_threshold,114.73
high_spend_rows,71
rows_out,7043


`TotalCharges` is text with blanks for `tenure == 0` customers. Those are genuine
zero-billing accounts, so they are filled with `0.0` instead of being dropped.


## 3. Target balance

In [4]:
counts = cleaned["Churn"].value_counts().sort_index()
print(f"churn rate = {cleaned['Churn'].mean():.1%}")
counts.rename(index={0: "retained", 1: "churned"}).to_frame("customers")


churn rate = 26.5%


,customers
Churn,
retained,5174
churned,1869


## 4. Numeric structure by churn

In [5]:
cleaned.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].agg(
    ["median", "mean"]
).round(1)


tenure       MonthlyCharges       TotalCharges        
      median  mean         median  mean       median    mean
Churn                                                       
0       38.0  37.6           64.4  61.3       1679.5  2549.9
1       10.0  18.0           79.6  74.4        703.6  1531.8

Churners have shorter tenure and higher monthly charges — the classic
high-price / low-switching-cost profile.

## 5. Categorical churn rates

In [6]:
def churn_table(column):
    return (
        cleaned.groupby(column)["Churn"]
        .agg(churn_rate="mean", customers="size")
        .sort_values("churn_rate", ascending=False)
        .assign(churn_rate=lambda d: (d["churn_rate"] * 100).round(1))
    )

for column in ["Contract", "InternetService", "PaymentMethod", "TechSupport"]:
    print(f"\n{column}")
    display(churn_table(column))



Contract


,churn_rate,customers
Contract,,
Month-to-month,42.7,3875
One year,11.3,1473
Two year,2.8,1695



InternetService


,churn_rate,customers
InternetService,,
Fiber optic,41.9,3096
DSL,19.0,2421
No,7.4,1526



PaymentMethod


,churn_rate,customers
PaymentMethod,,
Electronic check,45.3,2365
Mailed check,19.1,1612
Bank transfer (automatic),16.7,1544
Credit card (automatic),15.2,1522



TechSupport


,churn_rate,customers
TechSupport,,
No,41.6,3473
Yes,15.2,2044
No internet service,7.4,1526


## 6. Correlation structure

In [7]:
from churn.eda import _numeric_view
import seaborn as sns

corr = _numeric_view(cleaned).corr(numeric_only=True)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0,
            annot_kws={"size": 7}, square=True)
plt.title("Correlation matrix (ordinal-encoded categories)")
plt.show()


C:\Users\manis\AppData\Local\Temp\ipykernel_32536\3814704398.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Model results

Metrics are read from `reports/metrics.json`, written by `python run.py train`.

In [8]:
import json

metrics_path = ROOT / "reports" / "metrics.json"
metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}

if metrics:
    print("selected model:", metrics["selected_model"])
    print("threshold:", metrics["decision_threshold"])
    display(pd.DataFrame(metrics["cv_comparison"]).set_index("model").round(4))
    display(pd.Series(metrics["test_metrics"], name="hold-out").to_frame()
            .drop(index="confusion_matrix").round(4))
else:
    print("Run `python run.py train` first to populate metrics.")


selected model: logistic_regression
threshold: 0.6


,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,fit_seconds
model,,,,,,,,,,,,,
logistic_regression,0.8468,0.0118,0.6595,0.0166,0.7528,0.0128,0.5223,0.0163,0.8013,0.0321,0.6323,0.0195,0.63
gradient_boosting,0.8457,0.0110,0.6589,0.0201,0.8017,0.0103,0.6615,0.0258,0.5184,0.0314,0.5808,0.0256,8.24
random_forest,0.8411,0.0085,0.6465,0.0250,0.7867,0.0118,0.5854,0.0196,0.6709,0.0345,0.6251,0.0243,4.43


,hold-out
roc_auc,0.8448
average_precision,0.6519
accuracy,0.7715
precision,0.5539
recall,0.7139
f1,0.6238
brier,0.166


## 8. Takeaways

In [9]:
if metrics:
    top = pd.DataFrame(metrics["feature_importance"]).head(8)
    print("Top churn drivers:")
    for row in top.itertuples():
        print(f"  - {row.feature:<28} {row.importance:.4f}")


Top churn drivers:
  - InternetService_Fiber optic  1.1761
  - InternetService_DSL          1.0320
  - MonthlyCharges               0.8021
  - avg_monthly_spend            0.7862
  - contract_risk                0.5005
  - is_new_customer_0            0.4680
  - service_density              0.3656
  - num_services                 0.3656
